# Stage 2 BBO Challenge — Module 12: Week 1 Queries

**Imperial College London — Professional Certificate in ML & AI**  
**Author:** Srini Rajasekaran  
**Module:** 12 — Bayesian Optimisation  
**Round:** Week 1 of 13

---

## Overview

This notebook implements the Week 1 query strategy for all eight BBO challenge functions.
Two methods are evaluated for each function before a final query is selected:

1. **Multivariate OLS Regression** with full diagnostic assessment (R², Shapiro-Wilk, skewness, kurtosis, Durbin-Watson, p-values per coefficient)
2. **Gaussian Process surrogate with UCB acquisition** (beta=2.0, 8,000 candidate grid)

Regression is only used where four diagnostic gates are passed. Otherwise GP-UCB is used.

The approach extends the Module 12 taught content (Required Assignment 12.1 — UCB acquisition; Self-Study 12.1 and 12.2 — GP surrogate) to the real challenge functions.

---

In [ ]:
# Imports
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from scipy import stats

np.random.seed(42)
print('Libraries loaded.')

---
## Configuration

In [ ]:
# Function metadata
FUNCTION_META = {
    1: {'dims': 2, 'description': 'Radiation field detection',  'ls': 0.30},
    2: {'dims': 2, 'description': 'Noisy ML log-likelihood',    'ls': 0.30},
    3: {'dims': 3, 'description': 'Drug discovery',             'ls': 0.25},
    4: {'dims': 4, 'description': 'Warehouse allocation',       'ls': 0.20},
    5: {'dims': 4, 'description': 'Chemical yield',             'ls': 0.20},
    6: {'dims': 5, 'description': 'Recipe scoring',             'ls': 0.18},
    7: {'dims': 6, 'description': 'ML hyperparameter tuning',   'ls': 0.15},
    8: {'dims': 8, 'description': 'Neural network tuning',      'ls': 0.12},
}

# Diagnostic thresholds
R2_THRESHOLD  = 0.30   # Minimum R² for regression to be considered
SW_THRESHOLD  = 0.05   # Shapiro-Wilk p-value: above = normal residuals
DW_LOWER      = 1.50   # Durbin-Watson lower bound
DW_UPPER      = 2.50   # Durbin-Watson upper bound
UCB_BETA      = 2.0    # Exploration parameter: 2.0 = 95th percentile UCB
N_CANDIDATES  = 8000   # Random candidates evaluated per function

print(f'Config: R²>{R2_THRESHOLD} | SW p>{SW_THRESHOLD} | DW {DW_LOWER}-{DW_UPPER} | beta={UCB_BETA} | candidates={N_CANDIDATES:,}')

---
## Helper Functions

In [ ]:
def load_function(f_num, module='module_12'):
    """Load inputs and outputs for a given function."""
    base = f'../data/{module}/function_{f_num}/'
    X = np.load(base + 'initial_inputs.npy')
    y = np.load(base + 'initial_outputs.npy')
    return X, y

def regression_diagnostics(X, y):
    """Full OLS regression with diagnostic assessment."""
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X)
    lr = LinearRegression()
    lr.fit(X_sc, y)
    y_hat = lr.predict(X_sc)
    residuals = y - y_hat
    n, k = len(y), X.shape[1] + 1

    # Model fit
    r2 = float(r2_score(y, y_hat))
    r2_adj = 1 - (1 - r2) * (n - 1) / (n - k)
    rmse = float(np.sqrt(np.mean(residuals**2)))

    # Residual diagnostics
    sw_stat, sw_pval = stats.shapiro(residuals)
    skew = float(stats.skew(residuals))
    kurt = float(stats.kurtosis(residuals))
    dw = float(np.sum(np.diff(residuals)**2) / np.sum(residuals**2))

    # Standard errors and p-values
    X_aug = np.column_stack([np.ones(n), X_sc])
    XtX_inv = np.linalg.pinv(X_aug.T @ X_aug)
    mse = np.sum(residuals**2) / (n - k)
    se = np.sqrt(mse * np.diag(XtX_inv))
    beta_hat = XtX_inv @ X_aug.T @ y
    t_stats = beta_hat / (se + 1e-10)
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df=n-k))

    # Regression query: push significant betas in their direction
    # Insignificant betas (p>0.05) set to 0.50
    betas = lr.coef_
    beta_pvals = p_values[1:]  # exclude intercept
    reg_query = np.where(
        beta_pvals < 0.05,
        np.where(betas > 0, 0.95, 0.05),
        0.50
    )
    reg_pred = float(lr.predict(scaler.transform(reg_query.reshape(1,-1)))[0])

    # Diagnostic gates
    gate_r2      = r2 > R2_THRESHOLD
    gate_normal  = float(sw_pval) > SW_THRESHOLD
    gate_dw      = DW_LOWER < dw < DW_UPPER
    gate_improve = reg_pred > float(y.max())
    all_gates    = gate_r2 and gate_normal and gate_dw and gate_improve

    return {
        'r2': r2, 'r2_adj': r2_adj, 'rmse': rmse,
        'sw_pval': float(sw_pval), 'skew': skew, 'kurt': kurt, 'dw': dw,
        'betas': betas, 'beta_pvals': beta_pvals,
        'reg_query': reg_query, 'reg_pred': reg_pred,
        'gate_r2': gate_r2, 'gate_normal': gate_normal,
        'gate_dw': gate_dw, 'gate_improve': gate_improve,
        'all_gates': all_gates, 'scaler': scaler, 'model': lr
    }

def gp_ucb_query(X, y, length_scale, beta=UCB_BETA, n_candidates=N_CANDIDATES):
    """GP surrogate with UCB acquisition.
    UCB(x) = mu(x) + beta * sigma(x)
    beta=2.0 corresponds to 95th percentile of GP predictive distribution.
    """
    y_norm = (y - y.mean()) / (y.std() + 1e-8)
    kernel = RBF(length_scale=length_scale)
    gpr = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=3)
    gpr.fit(X, y_norm)

    grid = np.random.uniform(0, 1, (n_candidates, X.shape[1]))
    mu, sigma = gpr.predict(grid, return_std=True)
    ucb = mu + beta * sigma

    best_idx = np.argmax(ucb)
    return {
        'query': grid[best_idx],
        'mu': float(mu[best_idx]),
        'sigma': float(sigma[best_idx]),
        'ucb_score': float(ucb[best_idx]),
        'ucb_mean': float(ucb.mean()),
        'ucb_std': float(ucb.std()),
        'ucb_max': float(ucb.max()),
        'discrimination': 'High' if ucb.std() > 0.5 else ('Medium' if ucb.std() > 0.1 else 'Low'),
        'gpr': gpr, 'grid': grid, 'mu_all': mu, 'sigma_all': sigma, 'ucb_all': ucb
    }

def portal_format(x):
    """Format query point for portal submission."""
    return '-'.join([f'{v:.6f}' for v in x])

print('Helper functions ready.')

---
## Step 1: Initial Data Summary

In [ ]:
print(f'{"Fn":<5} {"Dims":<6} {"N obs":<8} {"Min y":<14} {"Max y":<14} {"Mean y":<14} {"Std y"}')
print('-' * 70)
for f in range(1, 9):
    X, y = load_function(f)
    print(f'F{f:<4} {FUNCTION_META[f]["dims"]:<6} {len(y):<8} {y.min():<14.6f} {y.max():<14.6f} {y.mean():<14.6f} {y.std():.6f}')

---
## Step 2: Regression Quality Assessment — All Functions

In [ ]:
print('REGRESSION DIAGNOSTIC ASSESSMENT')
print('=' * 90)
print(f'{"Fn":<5} {"R²":<8} {"Adj R²":<8} {"RMSE":<12} {"SW p":<10} {"Skew":<8} {"Kurt":<8} {"DW":<8} {"Gates":<6} {"Reg pred"}')
print('-' * 90)

reg_results = {}
for f in range(1, 9):
    X, y = load_function(f)
    d = regression_diagnostics(X, y)
    reg_results[f] = d
    gates = f'{int(d["gate_r2"])}{int(d["gate_normal"])}{int(d["gate_dw"])}{int(d["gate_improve"])}'
    verdict = 'PASS' if d['all_gates'] else 'FAIL'
    print(f'F{f:<4} {d["r2"]:<8.4f} {d["r2_adj"]:<8.4f} {d["rmse"]:<12.6f} '
          f'{d["sw_pval"]:<10.4f} {d["skew"]:<8.3f} {d["kurt"]:<8.3f} '
          f'{d["dw"]:<8.3f} {gates:<6} {verdict} (reg pred={d["reg_pred"]:.4f})')

print()
print('Gates: [R²>0.30][SW p>0.05][DW 1.5-2.5][Reg pred > best y]  1=pass 0=fail')

---
## Step 3: Coefficient p-values — F6 and F8 (Regression candidates only)

In [ ]:
for f in [6, 8]:
    X, y = load_function(f)
    d = reg_results[f]
    dims = FUNCTION_META[f]['dims']
    print(f'Function {f} — {FUNCTION_META[f]["description"]}')
    print(f'R²={d["r2"]:.4f} | SW p={d["sw_pval"]:.4f} | Skew={d["skew"]:.3f} | Kurt={d["kurt"]:.3f} | DW={d["dw"]:.4f}')
    print(f'{"Input":<10} {"Beta":<14} {"p-value":<12} {"Significant?":<14} {"Query direction"}')
    print('-' * 65)
    for i, (b, p) in enumerate(zip(d['betas'], d['beta_pvals'])):
        sig = '*** p<0.001' if p<0.001 else ('** p<0.01' if p<0.01 else ('* p<0.05' if p<0.05 else 'ns'))
        direction = ('Push to 0.95' if b>0 else 'Push to 0.05') if p<0.05 else 'Set to 0.50 (insignificant)'
        print(f'x{i+1:<9} {b:<14.6f} {p:<12.6f} {sig:<14} {direction}')
    print(f'Regression query: {portal_format(d["reg_query"])}')
    print(f'Predicted y: {d["reg_pred"]:.4f} vs current best: {y.max():.4f}')
    print()

---
## Step 4: GP-UCB Analysis — All Functions

In [ ]:
print(f'GP-UCB ANALYSIS — {N_CANDIDATES:,} candidates per function | beta={UCB_BETA}')
print('=' * 85)
print(f'{"Fn":<5} {"UCB mean":<12} {"UCB std":<12} {"UCB max":<12} {"Chosen mu":<12} {"Chosen sigma":<14} {"Discrimination"}')
print('-' * 85)

gp_results = {}
for f in range(1, 9):
    X, y = load_function(f)
    m = FUNCTION_META[f]
    gp = gp_ucb_query(X, y, length_scale=m['ls'])
    gp_results[f] = gp
    print(f'F{f:<4} {gp["ucb_mean"]:<12.4f} {gp["ucb_std"]:<12.4f} {gp["ucb_max"]:<12.4f} '
          f'{gp["mu"]:<12.4f} {gp["sigma"]:<14.4f} {gp["discrimination"]}')

---
## Step 5: GP Landscape Visualisation — Functions 1 and 2 (2D)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for row, f in enumerate([1, 2]):
    X, y = load_function(f)
    m = FUNCTION_META[f]
    gp = gp_results[f]

    g = np.linspace(0, 1, 60)
    G1, G2 = np.meshgrid(g, g)
    X_grid = np.column_stack([G1.ravel(), G2.ravel()])
    mu_g, sigma_g = gp['gpr'].predict(X_grid, return_std=True)
    ucb_g = mu_g + UCB_BETA * sigma_g
    query = gp['query']

    for col, (data, title, cmap) in enumerate([
        (mu_g, f'F{f}: GP Mean', 'viridis'),
        (sigma_g, f'F{f}: GP Uncertainty', 'plasma'),
        (ucb_g, f'F{f}: UCB (beta={UCB_BETA})', 'hot')
    ]):
        im = axes[row, col].contourf(G1, G2, data.reshape(60,60), levels=20, cmap=cmap)
        axes[row, col].scatter(X[:,0], X[:,1], c='white', s=50, zorder=5,
                               edgecolors='black', linewidth=0.8)
        axes[row, col].set_title(title, fontweight='bold', fontsize=10)
        plt.colorbar(im, ax=axes[row, col])

    axes[row, 2].scatter(query[0], query[1], c='cyan', s=250, zorder=10,
                         marker='*', edgecolors='black', linewidth=1.5,
                         label=f'Query ({query[0]:.3f}, {query[1]:.3f})')
    axes[row, 2].legend(fontsize=8)

plt.suptitle('GP Surrogate and UCB Acquisition — Functions 1 and 2\n'
             'Module 12 methodology applied to BBO challenge functions',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('week1_gp_landscape_f1_f2.png', dpi=130, bbox_inches='tight')
plt.show()
print('Saved: week1_gp_landscape_f1_f2.png')

---
## Step 6: Final Query Selection and Portal Submissions

In [ ]:
print('FINAL QUERY DECISIONS')
print('=' * 90)

final_queries = {}
for f in range(1, 9):
    X, y = load_function(f)
    d = reg_results[f]
    gp = gp_results[f]

    if d['all_gates']:
        method = 'Regression'
        query = d['reg_query']
        reason = f'R²={d["r2"]:.2f}, SW p={d["sw_pval"]:.2f}, DW={d["dw"]:.2f}, pred={d["reg_pred"]:.4f} > best={y.max():.4f}'
    else:
        method = 'GP-UCB'
        query = gp['query']
        # List which gates failed
        failed = []
        if not d['gate_r2']:      failed.append(f'R²={d["r2"]:.2f}<0.30')
        if not d['gate_normal']:  failed.append(f'SW p={d["sw_pval"]:.4f}<0.05')
        if not d['gate_dw']:      failed.append(f'DW={d["dw"]:.2f} outside 1.5-2.5')
        if not d['gate_improve']: failed.append(f'reg pred {d["reg_pred"]:.4f} < best {y.max():.4f}')
        reason = ' | '.join(failed)

    final_queries[f] = {'method': method, 'query': query, 'portal': portal_format(query)}

    print(f'F{f} ({FUNCTION_META[f]["dims"]}D) — {FUNCTION_META[f]["description"]}')
    print(f'  Method : {method}')
    print(f'  Reason : {reason}')
    print(f'  SUBMIT : {portal_format(query)}')
    print()

print('=' * 90)
print('PORTAL SUBMISSION SUMMARY')
print()
for f in range(1, 9):
    q = final_queries[f]
    print(f'Function {f} [{q["method"]:12s}]: {q["portal"]}')

---
## Step 7: UCB Score Distribution Summary

In [ ]:
print(f'UCB SCORE DISTRIBUTION ACROSS {N_CANDIDATES:,} CANDIDATES')
print(f'UCB(x) = GP mean(x) + {UCB_BETA} x GP std(x)')
print('=' * 75)
print(f'{"Fn":<5} {"Min":<10} {"P25":<10} {"Median":<10} {"Mean":<10} {"P75":<10} {"Max":<10} {"Std":<10} {"Level"}')
print('-' * 75)

for f in range(1, 9):
    gp = gp_results[f]
    ucb = gp['ucb_all']
    print(f'F{f:<4} {ucb.min():<10.4f} {np.percentile(ucb,25):<10.4f} '
          f'{np.percentile(ucb,50):<10.4f} {ucb.mean():<10.4f} '
          f'{np.percentile(ucb,75):<10.4f} {ucb.max():<10.4f} '
          f'{ucb.std():<10.4f} {gp["discrimination"]}')

print()
print('Discrimination: High (std>0.5) = GP has genuine landscape structure')
print('                Low  (std<0.1) = GP sees uniform uncertainty, query is blind exploration')

---
## Summary

| Function | Method | Rationale |
|----------|--------|-----------|
| F1 | GP-UCB | R²=0.04, non-normal residuals, zero UCB discrimination |
| F2 | GP-UCB | Regression does not predict improvement over current best |
| F3 | GP-UCB | DW=2.53 outside acceptable range |
| F4 | GP-UCB | Regression predicts worse than current best |
| F5 | GP-UCB | Outlier at y=1088 distorts regression, pred=788 < best |
| F6 | Regression | All gates passed. Only x4 (p=0.043) and x5 (p=0.017) significant |
| F7 | GP-UCB | Non-normal residuals (SW p=0.001) |
| F8 | Regression | All gates passed. R²=0.90. x1, x2, x3, x7 highly significant |

See `reflections/module_12_reflection.md` for full strategy narrative.